# CNN for MNIST Digit Classification: TensorFlow Guide

This notebook demonstrates building, training, and evaluating a Convolutional Neural Network (CNN) on the MNIST dataset using TensorFlow. The CNN architecture consists of two convolutional blocks followed by dense layers, achieving >98% accuracy on the test set.

**Learning Objectives:**
- Load and preprocess MNIST data
- Understand CNN architecture (convolution, pooling, dense layers)
- Train a model with validation monitoring
- Evaluate performance with multiple metrics
- Log experiments with MLflow for reproducibility

## 1. Setup & Imports

Import necessary libraries for data loading, model building, and visualization.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.datasets import mnist
from tensorflow.keras.utils import to_categorical
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import mlflow
import time

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

## 2. Load MNIST Data

The MNIST dataset contains 70,000 grayscale images of handwritten digits (28×28 pixels, 10 classes). We load it from Keras and visualize sample images.

In [ ]:
# Load MNIST dataset
(x_train_full, y_train_full), (x_test, y_test) = mnist.load_data()

# Split training data into train (50k) and validation (10k)
x_train, x_val = x_train_full[:50000], x_train_full[50000:]
y_train, y_val = y_train_full[:50000], y_train_full[50000:]

print(f"Training data shape: {x_train.shape}")
print(f"Validation data shape: {x_val.shape}")
print(f"Test data shape: {x_test.shape}")
print(f"Label shape: {y_train.shape}")

# Visualize sample images
fig, axes = plt.subplots(1, 10, figsize=(12, 2))
for i in range(10):
    axes[i].imshow(x_train[i], cmap='gray')
    axes[i].set_title(f"Label: {y_train[i]}")
    axes[i].axis('off')
plt.suptitle("Sample MNIST Images")
plt.tight_layout()
plt.show()

## 3. Data Preprocessing

Normalize pixel values to [0, 1] range and reshape to include channel dimension. Convert labels to one-hot encoding for multi-class classification.

In [ ]:
# Reshape and normalize
x_train = x_train.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_val = x_val.reshape(-1, 28, 28, 1).astype('float32') / 255.0
x_test = x_test.reshape(-1, 28, 28, 1).astype('float32') / 255.0

# One-hot encode labels
y_train_cat = to_categorical(y_train, 10)
y_val_cat = to_categorical(y_val, 10)
y_test_cat = to_categorical(y_test, 10)

print(f"Preprocessed training data shape: {x_train.shape}")
print(f"Preprocessed labels shape: {y_train_cat.shape}")
print(f"Pixel value range: [{x_train.min():.2f}, {x_train.max():.2f}]")

## 4. Model Definition

Build a CNN with two convolutional blocks. Each block applies convolution (learns spatial filters) followed by max pooling (downsamples). Dense layers perform final classification.

**Architecture:**
- **Conv Block 1**: Conv2D(32 filters, 3×3 kernel, ReLU) → MaxPool(2×2)
- **Conv Block 2**: Conv2D(64 filters, 3×3 kernel, ReLU) → MaxPool(2×2)
- **Dense Layers**: Flatten → Dense(128, ReLU) → Dropout(0.5) → Dense(10, Softmax)

This architecture reduces 28×28 images to 7×7 spatial dimensions while extracting features.

In [ ]:
# Set seed for reproducibility
tf.random.set_seed(42)

model = keras.Sequential([
    # Conv Block 1: Extract low-level features
    keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(28, 28, 1)),
    keras.layers.MaxPooling2D((2, 2)),  # 28×28 → 14×14
    
    # Conv Block 2: Extract higher-level features
    keras.layers.Conv2D(64, (3, 3), activation='relu'),
    keras.layers.MaxPooling2D((2, 2)),  # 14×14 → 7×7
    
    # Dense layers: Classification
    keras.layers.Flatten(),  # 64 × 7 × 7 = 3136 features
    keras.layers.Dense(128, activation='relu'),
    keras.layers.Dropout(0.5),  # Randomly drop 50% of neurons during training
    keras.layers.Dense(10, activation='softmax')  # Output: class probabilities
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## 5. Training Setup with MLflow

Configure MLflow for experiment tracking. We log hyperparameters before training and metrics after each epoch, enabling reproducibility and comparison with other frameworks.

In [ ]:
# Setup MLflow
mlflow.set_experiment("cnn_mnist_tensorflow")

# Define hyperparameters
hyperparams = {
    "learning_rate": 0.001,
    "batch_size": 32,
    "num_epochs": 10,  # Reduced for notebook demo
    "optimizer": "adam",
    "loss_function": "categorical_crossentropy",
    "num_conv_blocks": 2,
    "conv_filters_initial": 32,
    "dense_units": 128,
    "dropout_rate": 0.5,
    "random_seed": 42,
    "framework": "tensorflow"
}

print("Hyperparameters:")
for key, value in hyperparams.items():
    print(f"  {key}: {value}")

## 6. Training Loop

Train the model with MLflow logging. We monitor both training and validation loss/accuracy, enabling early detection of overfitting.

In [ ]:
with mlflow.start_run() as run:
    # Log hyperparameters
    mlflow.log_params(hyperparams)
    
    # Train model
    print(f"Training model for {hyperparams['num_epochs']} epochs...")
    start_time = time.time()
    
    history = model.fit(
        x_train, y_train_cat,
        validation_data=(x_val, y_val_cat),
        epochs=hyperparams['num_epochs'],
        batch_size=hyperparams['batch_size'],
        verbose=1
    )
    
    training_time = time.time() - start_time
    print(f"Training completed in {training_time:.2f} seconds")
    
    # Log per-epoch metrics to MLflow
    for epoch in range(len(history.history['loss'])):
        mlflow.log_metric("train_loss", float(history.history['loss'][epoch]), step=epoch)
        mlflow.log_metric("train_accuracy", float(history.history['accuracy'][epoch]), step=epoch)
        mlflow.log_metric("val_loss", float(history.history['val_loss'][epoch]), step=epoch)
        mlflow.log_metric("val_accuracy", float(history.history['val_accuracy'][epoch]), step=epoch)
    
    run_id = run.info.run_id
    print(f"MLflow Run ID: {run_id}")

## 7. Evaluation & Visualization

Evaluate the trained model on the test set and compute detailed metrics. Visualize training progress to understand model behavior.

In [ ]:
# Evaluate on test set
test_loss, test_accuracy = model.evaluate(x_test, y_test_cat, verbose=0)

# Get predictions for detailed metrics
y_pred_probs = model.predict(x_test, verbose=0)
y_pred = np.argmax(y_pred_probs, axis=1)
y_test_labels = np.argmax(y_test_cat, axis=1)

# Compute detailed metrics
precision = precision_score(y_test_labels, y_pred, average='macro')
recall = recall_score(y_test_labels, y_pred, average='macro')
f1 = f1_score(y_test_labels, y_pred, average='macro')

print(f"\n=== Test Set Performance ===")
print(f"Test Accuracy:  {test_accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall:    {recall:.4f}")
print(f"Test F1-Score:  {f1:.4f}")

# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(history.history['loss'], label='Training Loss', linewidth=2)
ax1.plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Model Loss Over Training')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
ax2.plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Model Accuracy Over Training')
ax2.set_ylim([0.9, 1.0])
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Model Persistence

Save the trained model to MLflow for retrieval and reuse. The model artifact can be loaded later for inference without retraining.

In [ ]:
# Save model to MLflow
mlflow.tensorflow.log_model(model, artifact_path="tensorflow_cnn_model")

print(f"Model saved to MLflow")
print(f"\nTo retrieve this model later:")
print(f"  mlflow.tensorflow.load_model('runs:/{run_id}/tensorflow_cnn_model')")

# Test loading and inference
sample_batch = x_test[:5]
predictions = model.predict(sample_batch, verbose=0)
predicted_classes = np.argmax(predictions, axis=1)
true_classes = y_test[:5]

print(f"\nSample Predictions:")
print(f"Predicted: {predicted_classes}")
print(f"True:      {true_classes}")